In [18]:
# Experiment 1:

from collections import deque

graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'E'],
    'D': ['B', 'G'],
    'E': ['B', 'C', 'F'],
    'F': ['E', 'H'],
    'G': ['D', 'H'],
    'H': ['F', 'G']
}

def bfs(graph, start, goal):
    queue = deque([start])
    visited = {start}
    parent = {start: None}
    distance = {start: 0}
    order = []

    print("BFS: Hospital Navigation")
    print(f"Start: {start} | Goal: {goal}\n")

    while queue:
        node = queue.popleft()
        order.append(node)

        print(f"Step {len(order)}: Node = {node}")
        print(f"  Queue: {list(queue)}")

        if node == goal:
            print("  Goal reached!")
            break

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = node
                distance[neighbor] = distance[node] + 1
                queue.append(neighbor)
                print(f"  {node} -> {neighbor}: Added to queue")
            else:
                print(f"  {node} -> {neighbor}: Already visited")

        print(f"  Visited: {sorted(visited)}")

    if goal not in parent:
        print("\nNo route found.")
        return

    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    print("\n--- Final Result ---")
    print("Traversal order:", " -> ".join(order))
    print("Shortest path  :", " -> ".join(path))
    print("Path cost      :", distance[goal], "corridors")
    print("Nodes explored :", len(order))


bfs(graph, 'A', 'H')


BFS: Hospital Navigation
Start: A | Goal: H

Step 1: Node = A
  Queue: []
  A -> B: Added to queue
  A -> C: Added to queue
  Visited: ['A', 'B', 'C']
Step 2: Node = B
  Queue: ['C']
  B -> A: Already visited
  B -> D: Added to queue
  B -> E: Added to queue
  Visited: ['A', 'B', 'C', 'D', 'E']
Step 3: Node = C
  Queue: ['D', 'E']
  C -> A: Already visited
  C -> E: Already visited
  Visited: ['A', 'B', 'C', 'D', 'E']
Step 4: Node = D
  Queue: ['E']
  D -> B: Already visited
  D -> G: Added to queue
  Visited: ['A', 'B', 'C', 'D', 'E', 'G']
Step 5: Node = E
  Queue: ['G']
  E -> B: Already visited
  E -> C: Already visited
  E -> F: Added to queue
  Visited: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Step 6: Node = G
  Queue: ['F']
  G -> D: Already visited
  G -> H: Added to queue
  Visited: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
Step 7: Node = F
  Queue: ['H']
  F -> E: Already visited
  F -> H: Already visited
  Visited: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
Step 8: Node = H
  Queue: 

In [19]:

# Experiment 2: DFS for Dependency and Reachability Analysis
# Directed adjacency list representing the AI pipeline
graph = {
    'A': ['B', 'C', 'D'],
    'B': ['E'],
    'C': ['F'],
    'D': ['G'],
    'E': ['H'],
    'F': ['H'],
    'G': ['H'],
    'H': ['L'],
    'L': []
}

def dfs(graph, start, target):
    visited = set()
    traversal = []
    parent = {start: None}

    def explore(node):
        visited.add(node)
        traversal.append(node)

        if node == target:
            return True

        for neighbour in graph.get(node, []):
            if neighbour not in visited:
                parent[neighbour] = node

                if explore(neighbour):
                    return True

        return False

    found = explore(start)

    print("\nDFS Traversal Order:", " -> ".join(traversal))

    if found:
        path = []
        node = target

        while node is not None:
            path.append(node)
            node = parent[node]

        path.reverse()

        print("Target Reachable: Yes")
        print("Valid Path:", " -> ".join(path))
    else:
        print("Target Reachable: No")
        print("No valid path exists.")

# Accept input from the user
start = input("Enter starting task (A, B, C, D, E, F, G, H, L): ").upper()
target = input("Enter target task (A, B, C, D, E, F, G, H, L): ").upper()

if start not in graph or target not in graph:
    print("Invalid task! Please enter a valid task label.")
else:
    dfs(graph, start, target)


Enter starting task (A, B, C, D, E, F, G, H, L): A
Enter target task (A, B, C, D, E, F, G, H, L): L

DFS Traversal Order: A -> B -> E -> H -> L
Target Reachable: Yes
Valid Path: A -> B -> E -> H -> L


In [20]:
# Experiment-3:
from collections import deque

# Capacities of the two containers
CAP_A = 4
CAP_B = 3

# State: (water in 4-litre container, water in 3-litre container)
INITIAL_STATE = (0, 0)
GOAL_AMOUNT = 2


def get_next_states(state):
    a, b = state
    next_states = []

    def add(action, new_a, new_b):
        new_state = (new_a, new_b)
        if new_state != state:
            next_states.append((action, new_state))

    # Fill either container
    add("Fill 4-litre container", CAP_A, b)
    add("Fill 3-litre container", a, CAP_B)

    # Empty either container
    add("Empty 4-litre container", 0, b)
    add("Empty 3-litre container", a, 0)

    # Pour from 4-litre container into 3-litre container
    transfer = min(a, CAP_B - b)
    add("Pour 4L into 3L", a - transfer, b + transfer)

    # Pour from 3-litre container into 4-litre container
    transfer = min(b, CAP_A - a)
    add("Pour 3L into 4L", a + transfer, b - transfer)

    return next_states


def bfs():
    queue = deque([INITIAL_STATE])
    visited = {INITIAL_STATE}
    parent = {INITIAL_STATE: (None, None)}

    while queue:
        state = queue.popleft()
        a, b = state

        # Goal test
        if a == GOAL_AMOUNT:
            solution = []
            current = state

            while parent[current][0] is not None:
                previous, action = parent[current]
                solution.append((action, current))
                current = previous

            solution.reverse()
            return solution

        for action, next_state in get_next_states(state):
            if next_state not in visited:
                visited.add(next_state)
                parent[next_state] = (state, action)
                queue.append(next_state)

    return None


# Execute BFS
solution = bfs()

if solution is None:
    print("No solution exists.")
else:
    print("Initial State: (0, 0)")
    print("State format: (4-litre container, 3-litre container)")
    print()

    current_state = INITIAL_STATE

    for step, (action, next_state) in enumerate(solution, start=1):
        print(f"Step {step}: {action}")
        print(f"        {current_state} -> {next_state}")
        current_state = next_state

    print("\nGoal reached!")
    print(f"4-litre container contains {current_state[0]} litres.")
    print(f"Total operations: {len(solution)}")


Initial State: (0, 0)
State format: (4-litre container, 3-litre container)

Step 1: Fill 4-litre container
        (0, 0) -> (4, 0)
Step 2: Pour 4L into 3L
        (4, 0) -> (1, 3)
Step 3: Empty 3-litre container
        (1, 3) -> (1, 0)
Step 4: Pour 4L into 3L
        (1, 0) -> (0, 1)
Step 5: Fill 4-litre container
        (0, 1) -> (4, 1)
Step 6: Pour 4L into 3L
        (4, 1) -> (2, 3)

Goal reached!
4-litre container contains 2 litres.
Total operations: 6


In [21]:
# Experiment-4:
import heapq

# Weighted graph representing the delivery network
graph = {
    'A': {'B': 4, 'C': 3},
    'B': {'A': 4, 'C': 2, 'D': 5},
    'C': {'A': 3, 'B': 2, 'D': 6, 'E': 4},
    'D': {'B': 5, 'C': 6, 'E': 1, 'F': 2},
    'E': {'C': 4, 'D': 1, 'F': 3},
    'F': {'D': 2, 'E': 3}
}

# Heuristic estimates for reaching the goal
heuristic = {
    'A': 6,
    'B': 4,
    'C': 5,
    'D': 2,
    'E': 3,
    'F': 0
}


def a_star(start, goal):
    # Priority queue stores (f, g, node)
    open_list = [(heuristic[start], 0, start)]

    # Lowest known cost from start to each node
    g_cost = {start: 0}

    # Parent pointers for reconstructing the route
    parent = {start: None}

    # Nodes whose expansion is reported
    expanded = []

    while open_list:
        f, g, current = heapq.heappop(open_list)

        # Ignore outdated queue entries
        if g != g_cost.get(current):
            continue

        expanded.append(current)
        print(
            f"Expanding {current}: "
            f"g(n)={g}, h(n)={heuristic[current]}, f(n)={f}"
        )

        # Goal test
        if current == goal:
            path = []
            node = goal

            while node is not None:
                path.append(node)
                node = parent[node]

            path.reverse()

            print("\nOptimal route:", " -> ".join(path))
            print("Total path cost:", g_cost[goal])
            print("Expansion order:", " -> ".join(expanded))
            return path, g_cost[goal], expanded

        # Explore neighbouring locations
        for neighbour, cost in graph[current].items():
            new_g = g_cost[current] + cost

            if new_g < g_cost.get(neighbour, float('inf')):
                g_cost[neighbour] = new_g
                parent[neighbour] = current
                new_f = new_g + heuristic[neighbour]

                heapq.heappush(
                    open_list, (new_f, new_g, neighbour)
                )

    print("No route exists.")
    return None, float('inf'), expanded


# Get start and destination from the user
start = input("Enter starting location (A-F): ").upper()
goal = input("Enter destination location (A-F): ").upper()

if start not in graph or goal not in graph:
    print("Invalid location. Enter a node from A to F.")
else:
    a_star(start, goal)


Enter starting location (A-F): A
Enter destination location (A-F): F
Expanding A: g(n)=0, h(n)=6, f(n)=6
Expanding C: g(n)=3, h(n)=5, f(n)=8
Expanding B: g(n)=4, h(n)=4, f(n)=8
Expanding E: g(n)=7, h(n)=3, f(n)=10
Expanding D: g(n)=8, h(n)=2, f(n)=10
Expanding F: g(n)=10, h(n)=0, f(n)=10

Optimal route: A -> C -> E -> F
Total path cost: 10
Expansion order: A -> C -> B -> E -> D -> F


In [22]:

# Experiment 5: 8-Queens Problem Using Backtracking

N = 8
board = [-1] * N
placement_attempts = 0
backtracks = 0


# Check whether a queen can be placed safely
def is_safe(row, col):
    for prev_row in range(row):
        prev_col = board[prev_row]

        # Check same column and diagonal conflicts
        if prev_col == col:
            return False

        if abs(prev_col - col) == abs(prev_row - row):
            return False

    return True


# Solve using recursive backtracking
def solve(row):
    global placement_attempts, backtracks

    # All eight queens have been placed
    if row == N:
        return True

    # Try every column in the current row
    for col in range(N):
        placement_attempts += 1

        if is_safe(row, col):
            board[row] = col

            if solve(row + 1):
                return True

            # Undo the placement and try another column
            board[row] = -1
            backtracks += 1

    return False


# Display the chessboard
def display_board():
    print("\nFinal Chessboard:")
    print("  " + " ".join(str(c + 1) for c in range(N)))

    for row in range(N):
        print(row + 1, end=" ")
        for col in range(N):
            if board[row] == col:
                print("Q", end=" ")
            else:
                print(".", end=" ")
        print()


# Execute the program
if solve(0):
    display_board()

    print("\nQueen Positions (Row, Column):")
    for row in range(N):
        print(f"Queen {row + 1}: ({row + 1}, {board[row] + 1})")

    print("\nPlacement attempts:", placement_attempts)
    print("Backtracks:", backtracks)
    print("Result: Valid arrangement found.")
else:
    print("No solution exists.")



Final Chessboard:
  1 2 3 4 5 6 7 8
1 Q . . . . . . . 
2 . . . . Q . . . 
3 . . . . . . . Q 
4 . . . . . Q . . 
5 . . Q . . . . . 
6 . . . . . . Q . 
7 . Q . . . . . . 
8 . . . Q . . . . 

Queen Positions (Row, Column):
Queen 1: (1, 1)
Queen 2: (2, 5)
Queen 3: (3, 8)
Queen 4: (4, 6)
Queen 5: (5, 3)
Queen 6: (6, 7)
Queen 7: (7, 2)
Queen 8: (8, 4)

Placement attempts: 876
Backtracks: 105
Result: Valid arrangement found.


In [23]:
# Experiment-6:
# Map Colouring Problem using CSP and Backtracking

regions = ['A', 'B', 'C', 'D', 'E', 'F']

colours = ['Red', 'Green', 'Blue']

neighbours = {
    'A': ['B', 'C'],
    'B': ['A', 'C', 'D'],
    'C': ['A', 'B', 'D', 'E'],
    'D': ['B', 'C', 'E', 'F'],
    'E': ['C', 'D', 'F'],
    'F': ['D', 'E']
}

assignment = {}
attempts = 0

def is_valid(region, colour):
    for neighbour in neighbours[region]:
        if neighbour in assignment and assignment[neighbour] == colour:
            return False
    return True

def backtracking():
    global attempts

    if len(assignment) == len(regions):
        return True

    for region in regions:
        if region not in assignment:
            break

    for colour in colours:
        attempts += 1

        if is_valid(region, colour):
            assignment[region] = colour

            if backtracking():
                return True

            del assignment[region]

    return False

if backtracking():
    print("Valid Colour Assignment:")
    for region in regions:
        print(region, ":", assignment[region])

    print("All adjacency constraints are satisfied.")
else:
    print("No valid colour assignment exists.")

print("Number of assignments attempted:", attempts)


Valid Colour Assignment:
A : Red
B : Green
C : Blue
D : Red
E : Green
F : Blue
All adjacency constraints are satisfied.
Number of assignments attempted: 12


In [24]:
tree = [3, 5, 6, 9, 1, 2, 0, -1]
def minimax(values, depth, maximizing):
    if depth == 3:
        return values.pop(0)
    if maximizing:
        left = minimax(values, depth + 1, False)
        right = minimax(values, depth + 1, False)
        value = max(left, right)
        print("MAX node value:", value)
        return value
    else:
        left = minimax(values, depth + 1, True)
        right = minimax(values, depth + 1, True)
        value = min(left, right)
        print("MIN node value:", value)
        return value
# Use a copy because the recursive function consumes the list.
root_value = minimax(tree.copy(), 0, True)
print("Best utility for AI:", root_value)
print("Best move: Left subtree" if root_value == 5 else "Best move: Right subtree")

MAX node value: 5
MAX node value: 9
MIN node value: 5
MAX node value: 2
MAX node value: 0
MIN node value: 0
MAX node value: 5
Best utility for AI: 5
Best move: Left subtree


In [25]:
# Experiment-7:
# Binary game tree with eight terminal utility values
tree = [3, 5, 6, 9, 1, 2, 0, -1]
leaf_index = 0
nodes_evaluated = 0
branches_pruned = 0
def alpha_beta(depth, index, maximizing, alpha, beta):
    global nodes_evaluated, branches_pruned
    if depth == 3:
        nodes_evaluated += 1
        value = tree[index]
        print("Evaluated leaf:", value)
        return value
    if maximizing:
        value = float("-inf")
        for child in range(2):
            child_value = alpha_beta(
                depth + 1, index * 2 + child,
                False, alpha, beta
            )
            value = max(value, child_value)
            alpha = max(alpha, value)
            if alpha >= beta:
                branches_pruned += 1
                # Count the remaining child subtree as pruned.
                branches_pruned += 2 ** (3 - (depth + 1))
                break
        return value
    else:
        value = float("inf")
        for child in range(2):
            child_value = alpha_beta(
                depth + 1, index * 2 + child,
                True, alpha, beta
            )
            value = min(value, child_value)
            beta = min(beta, value)
            if alpha >= beta:
                # Remaining child subtree is skipped.
                branches_pruned += 2 ** (3 - (depth + 1))
                break
        return value
best_value = alpha_beta(0, 0, True, float("-inf"), float("inf"))
print("Best move: Left subtree" if best_value == 5 else "Best move: Right subtree")
print("Best utility value:", best_value)
print("Terminal nodes evaluated:", nodes_evaluated)
print("Branches/nodes skipped by pruning:", branches_pruned)

Evaluated leaf: 3
Evaluated leaf: 5
Evaluated leaf: 6
Evaluated leaf: 1
Evaluated leaf: 2
Best move: Left subtree
Best utility value: 5
Terminal nodes evaluated: 5
Branches/nodes skipped by pruning: 4


In [26]:
# Experiment-8:
# Forward Chaining for Medical Appointment Eligibility

# Step 1: Define initial facts
facts = {
    "patient_registered",
    "has_fever",
    "has_high_blood_pressure",
    "emergency_symptoms"
}

# Step 2: Define inference rules
rules = [
    ({"patient_registered", "has_fever"}, "needs_medical_checkup"),
    ({"has_high_blood_pressure"}, "requires_monitoring"),
    ({"needs_medical_checkup", "requires_monitoring"},
     "priority_consultation"),
    ({"emergency_symptoms"}, "emergency_priority"),
    ({"priority_consultation", "patient_registered"},
     "eligible_for_priority_appointment")
]

# Step 3: Implement Forward Chaining
def forward_chaining(facts, rules):

    print("Initial Facts:")
    for fact in facts:
        print("-", fact)

    inferred_facts = set()
    new_fact_found = True

    while new_fact_found:
        new_fact_found = False

        for conditions, conclusion in rules:

            if conditions.issubset(facts):
                if conclusion not in facts:

                    facts.add(conclusion)
                    inferred_facts.add(conclusion)
                    new_fact_found = True

                    print("\nNewly Inferred Fact:", conclusion)

    # Step 4: Display final facts
    print("\nFinal Set of Facts:")
    for fact in facts:
        print("-", fact)

    print("\nNewly Inferred Facts:")
    for fact in inferred_facts:
        print("-", fact)


# Step 5: Execute the program
forward_chaining(facts, rules)

Initial Facts:
- has_fever
- has_high_blood_pressure
- emergency_symptoms
- patient_registered

Newly Inferred Fact: needs_medical_checkup

Newly Inferred Fact: requires_monitoring

Newly Inferred Fact: priority_consultation

Newly Inferred Fact: emergency_priority

Newly Inferred Fact: eligible_for_priority_appointment

Final Set of Facts:
- eligible_for_priority_appointment
- has_fever
- emergency_symptoms
- requires_monitoring
- needs_medical_checkup
- emergency_priority
- patient_registered
- has_high_blood_pressure
- priority_consultation

Newly Inferred Facts:
- eligible_for_priority_appointment
- requires_monitoring
- needs_medical_checkup
- emergency_priority
- priority_consultation


In [27]:
#EX-9
facts = {
    "fever",
    "cough",
    "high_temperature",
    "age_over_65"
}

rules = [
    (["fever", "cough"], "respiratory_symptoms"),
    (["respiratory_symptoms", "high_temperature"],
     "suspected_infection"),
    (["suspected_infection", "age_over_65"],
     "priority_consultation")
]

trace = []

print("Forward Chaining Using Facts and Rules\n")

print("Initial Facts:")
for fact in sorted(facts):
    print("-", fact)

changed = True

while changed:
    changed = False

    for conditions, conclusion in rules:
        if all(condition in facts for condition in conditions):
            if conclusion not in facts:
                facts.add(conclusion)
                changed = True
                trace.append(
                    f"IF {conditions} THEN {conclusion}"
                )
                print(f"\nRule applied: {trace[-1]}")
                print(f"New fact inferred: {conclusion}")

print("\n--- Final Facts ---")
for fact in sorted(facts):
    print("-", fact)

print("\n--- Final Conclusion ---")
if "priority_consultation" in facts:
    print("Patient qualifies for a priority consultation.")
else:
    print("Priority consultation criteria not established.")

print("\nTotal new facts inferred:", len(trace))

Forward Chaining Using Facts and Rules

Initial Facts:
- age_over_65
- cough
- fever
- high_temperature

Rule applied: IF ['fever', 'cough'] THEN respiratory_symptoms
New fact inferred: respiratory_symptoms

Rule applied: IF ['respiratory_symptoms', 'high_temperature'] THEN suspected_infection
New fact inferred: suspected_infection

Rule applied: IF ['suspected_infection', 'age_over_65'] THEN priority_consultation
New fact inferred: priority_consultation

--- Final Facts ---
- age_over_65
- cough
- fever
- high_temperature
- priority_consultation
- respiratory_symptoms
- suspected_infection

--- Final Conclusion ---
Patient qualifies for a priority consultation.

Total new facts inferred: 3


In [28]:
#EX-10
clauses = [
    {"EnjoysLearning"},                  # C1
    {"HasFreeTime"},                     # C2
    {"NotEnjoysLearning", "Happy"},      # C3
    {"NotHasFreeTime", "Happy"}          # C4
]

goal = "Happy"

print("Resolution Refutation\n")

print("--- Knowledge Base (CNF) ---")

for i, clause in enumerate(clauses, start=1):
    print(f"C{i}: {' OR '.join(sorted(clause))}")

# Negate the goal and add it to the knowledge base

negated_goal = "Not" + goal

clauses.append({negated_goal})

print(f"\nNegated goal added: {negated_goal}")

# Resolve complementary literals

derived = set()

empty_clause_found = False

print("\n--- Resolution Steps ---")

changed = True

while changed and not empty_clause_found:

    changed = False

    current_clauses = list(clauses)

    for i in range(len(current_clauses)):

        for j in range(i + 1, len(current_clauses)):

            c1 = current_clauses[i]

            c2 = current_clauses[j]

            for literal in c1:

                if literal.startswith("Not"):
                    complement = literal[3:]
                else:
                    complement = "Not" + literal

                if complement in c2:

                    resolvent = (c1 - {literal}) | (c2 - {complement})

                    print(
                        f"Resolve {sorted(c1)} and {sorted(c2)} "
                        f"on {literal} -> {sorted(resolvent)}"
                    )

                    if not resolvent:
                        empty_clause_found = True
                        break

                    frozen_resolvent = frozenset(resolvent)

                    if (frozen_resolvent not in
                            {frozenset(c) for c in clauses} and
                        frozen_resolvent not in derived):

                        derived.add(frozen_resolvent)

                        changed = True

            if empty_clause_found:
                break

        if empty_clause_found:
            break

    for clause in derived:
        new_clause = set(clause)

        if new_clause not in clauses:
            clauses.append(new_clause)

    derived.clear()

print("\n--- Final Result ---")

if empty_clause_found:
    print("Empty clause derived: Yes")
    print("Contradiction found.")
    print("Therefore, the goal 'Happy' is proved.")

else:
    print("Empty clause derived: No")
    print("The goal could not be proved by this resolution run.")

Resolution Refutation

--- Knowledge Base (CNF) ---
C1: EnjoysLearning
C2: HasFreeTime
C3: Happy OR NotEnjoysLearning
C4: Happy OR NotHasFreeTime

Negated goal added: NotHappy

--- Resolution Steps ---
Resolve ['EnjoysLearning'] and ['Happy', 'NotEnjoysLearning'] on EnjoysLearning -> ['Happy']
Resolve ['HasFreeTime'] and ['Happy', 'NotHasFreeTime'] on HasFreeTime -> ['Happy']
Resolve ['Happy', 'NotEnjoysLearning'] and ['NotHappy'] on Happy -> ['NotEnjoysLearning']
Resolve ['Happy', 'NotHasFreeTime'] and ['NotHappy'] on Happy -> ['NotHasFreeTime']
Resolve ['EnjoysLearning'] and ['Happy', 'NotEnjoysLearning'] on EnjoysLearning -> ['Happy']
Resolve ['EnjoysLearning'] and ['NotEnjoysLearning'] on EnjoysLearning -> []

--- Final Result ---
Empty clause derived: Yes
Contradiction found.
Therefore, the goal 'Happy' is proved.


In [29]:
#EX-11
parents = [
    ("Ravi", "Arun"),
    ("Ravi", "Meena"),
    ("Arun", "Kiran"),
    ("Meena", "Priya"),
    ("Suresh", "Ravi")
]

def is_parent(parent, child):
    return (parent, child) in parents

def get_children(parent):
    return [child for p, child in parents if p == parent]

def get_parents(child):
    return [parent for parent, c in parents if c == child]

def get_grandparents(person):
    grandparents = []

    for parent in get_parents(person):
        grandparents.extend(get_parents(parent))

    return grandparents

# Display family relationships

person = "Kiran"

print("Family Tree Relationships")
print("-------------------------")

print("Parents of", person, ":", get_parents(person))

print("Children of Ravi:", get_children("Ravi"))

print("Grandparents of", person, ":", get_grandparents(person))

print("\nIs Ravi a parent of Arun?", is_parent("Ravi", "Arun"))

print("Is Suresh a grandparent of Kiran?",
      "Suresh" in get_grandparents("Kiran"))

Family Tree Relationships
-------------------------
Parents of Kiran : ['Arun']
Children of Ravi: ['Arun', 'Meena']
Grandparents of Kiran : ['Ravi']

Is Ravi a parent of Arun? True
Is Suresh a grandparent of Kiran? False


In [30]:
#EX-12
import re

# Healthcare symptom patterns

symptom_patterns = {

    "Emergency Warning Signs": [
        r"\bchest pain\b",
        r"\bdifficulty breathing\b",
        r"\bshortness of breath\b",
        r"\bface drooping\b",
        r"\bsudden weakness\b",
        r"\bsevere bleeding\b"
    ],

    "Respiratory Symptoms": [
        r"\bcough\b",
        r"\bsore throat\b",
        r"\brunny nose\b",
        r"\bfever\b"
    ],

    "Digestive Symptoms": [
        r"\bnausea\b",
        r"\bvomiting\b",
        r"\bstomach pain\b",
        r"\babdominal pain\b",
        r"\bdiarrhea\b"
    ],

    "Neurological or General Symptoms": [
        r"\bheadache\b",
        r"\bdizziness\b",
        r"\bfatigue\b",
        r"\bweakness\b"
    ]

}

# Guidance associated with matched patterns

guidance = {

    "Emergency Warning Signs":
        "Seek emergency medical care immediately. Do not delay care "
        "while using this system.",

    "Respiratory Symptoms":
        "Consider professional medical advice if symptoms persist, "
        "worsen, or are accompanied by breathing difficulty.",

    "Digestive Symptoms":
        "Maintain hydration if appropriate and seek medical advice "
        "for persistent or worsening symptoms.",

    "Neurological or General Symptoms":
        "Consider medical assessment if symptoms persist or worsen. "
        "Sudden weakness or other sudden neurological changes require "
        "emergency assessment."

}


def match_symptoms(patient_input):

    text = patient_input.lower()

    matched_categories = []
    matched_symptoms = []

    for category, patterns in symptom_patterns.items():

        category_matches = []

        for pattern in patterns:

            if re.search(pattern, text):
                category_matches.append(
                    re.search(pattern, text).group()
                )

        if category_matches:
            matched_categories.append(category)
            matched_symptoms.extend(category_matches)

    return matched_categories, matched_symptoms


def healthcare_assistant():

    print("HEALTHCARE SYMPTOM ASSESSMENT SYSTEM")
    print("-" * 40)

    print("Please enter your symptoms:")
    print("Example: Fever, cough, headache and sore throat.")

    patient_input = input("\nEnter your symptoms: ").strip()

    if not patient_input:

        print("System: Please enter a symptom description.")
        return

    categories, symptoms = match_symptoms(patient_input)

    print("\nASSESSMENT REPORT")
    print("-" * 40)

    if not categories:

        print("Matched symptoms: None")
        print("System: No predefined pattern was matched.")

        print("Please describe the symptoms more clearly or "
              "consult a healthcare professional.")

        return

    print("Matched symptoms:", ", ".join(sorted(set(symptoms))))
    print("Matched categories:", ", ".join(categories))

    # Emergency patterns take priority over other matches

    if "Emergency Warning Signs" in categories:

        print("\nPriority: EMERGENCY WARNING SIGN")
        print("Guidance:", guidance["Emergency Warning Signs"])

    else:

        print("\nPriority: NON-EMERGENCY PATTERN MATCH")

        for category in categories:
            print(f"{category}: {guidance[category]}")

    print("\nNote: This system does not diagnose diseases "
          "or replace professional medical assessment.")


healthcare_assistant()

HEALTHCARE SYMPTOM ASSESSMENT SYSTEM
----------------------------------------
Please enter your symptoms:
Example: Fever, cough, headache and sore throat.

Enter your symptoms: fever,cough,headache

ASSESSMENT REPORT
----------------------------------------
Matched symptoms: cough, fever, headache
Matched categories: Respiratory Symptoms, Neurological or General Symptoms

Priority: NON-EMERGENCY PATTERN MATCH
Respiratory Symptoms: Consider professional medical advice if symptoms persist, worsen, or are accompanied by breathing difficulty.
Neurological or General Symptoms: Consider medical assessment if symptoms persist or worsen. Sudden weakness or other sudden neurological changes require emergency assessment.

Note: This system does not diagnose diseases or replace professional medical assessment.


In [31]:
#EX-13

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target.reshape(-1, 1)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Neural network architecture
input_neurons = X_train.shape[1]
hidden_neurons = 8
output_neurons = 1

np.random.seed(42)

W1 = np.random.randn(input_neurons, hidden_neurons) * np.sqrt(2 / input_neurons)
b1 = np.zeros((1, hidden_neurons))

W2 = np.random.randn(hidden_neurons, output_neurons) * np.sqrt(2 / hidden_neurons)
b2 = np.zeros((1, output_neurons))

# Sigmoid activation function
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

# Training parameters
learning_rate = 0.05
epochs = 3000

# Backpropagation training
for epoch in range(epochs):

    hidden_input = np.dot(X_train, W1) + b1
    hidden_output = sigmoid(hidden_input)

    final_input = np.dot(hidden_output, W2) + b2
    predicted_output = sigmoid(final_input)

    output_delta = (predicted_output - y_train) / len(y_train)

    dW2 = np.dot(hidden_output.T, output_delta)
    db2 = np.sum(output_delta, axis=0, keepdims=True)

    hidden_delta = (
        np.dot(output_delta, W2.T)
        * hidden_output * (1 - hidden_output)
    )

    dW1 = np.dot(X_train.T, hidden_delta)
    db1 = np.sum(hidden_delta, axis=0, keepdims=True)

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

# Testing the neural network
hidden_test = sigmoid(np.dot(X_test, W1) + b1)
test_output = sigmoid(np.dot(hidden_test, W2) + b2)

y_pred = (test_output >= 0.5).astype(int)

# Model evaluation
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Artificial Neural Network using Backpropagation")
print("-----------------------------------------------")
print("Input neurons:", input_neurons)
print("Hidden neurons:", hidden_neurons)
print("Output neurons:", output_neurons)
print("Training epochs:", epochs)

print("\nClassification Accuracy:",
      round(accuracy * 100, 2), "%")

print("\nConfusion Matrix:")
print(cm)

print("\nClass Labels:")
print("0 - Malignant")
print("1 - Benign")


Artificial Neural Network using Backpropagation
-----------------------------------------------
Input neurons: 30
Hidden neurons: 8
Output neurons: 1
Training epochs: 3000

Classification Accuracy: 95.61 %

Confusion Matrix:
[[41  1]
 [ 4 68]]

Class Labels:
0 - Malignant
1 - Benign


In [32]:
#Exper-14:
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Generate sample data
x_train = np.random.rand(1000, 784).astype("float32")
y_train = np.random.randint(0, 10, 1000)

x_test = np.random.rand(200, 784).astype("float32")
y_test = np.random.randint(0, 10, 200)

# One-hot encoding
y_train_encoded = tf.keras.utils.to_categorical(y_train, 10)
y_test_encoded = tf.keras.utils.to_categorical(y_test, 10)

# Build ANN model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="sigmoid"),
    tf.keras.layers.Dense(10, activation="softmax")
])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Handwritten Digit Recognition")
print("--------------------------------")

model.summary()

# Train the model
history = model.fit(
    x_train,
    y_train_encoded,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# Predict test data
predicted_probabilities = model.predict(x_test, verbose=0)

y_pred = np.argmax(predicted_probabilities, axis=1)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nModel Evaluation")
print("--------------------------------")

print("Testing Accuracy: {:.2f}%".format(accuracy * 100))

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Handwritten Digit Recognition
--------------------------------


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.1122 - loss: 2.4013 - val_accuracy: 0.0700 - val_loss: 2.3966
Epoch 2/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0933 - loss: 2.3317 - val_accuracy: 0.1700 - val_loss: 2.3207
Epoch 3/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1056 - loss: 2.3310 - val_accuracy: 0.0700 - val_loss: 2.3304
Epoch 4/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1656 - loss: 2.2606 - val_accuracy: 0.1200 - val_loss: 2.3054
Epoch 5/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1889 - loss: 2.2454 - val_accuracy: 0.1200 - val_loss: 2.3063

Model Evaluation
--------------------------------
Testing Accuracy: 9.50%

Confusion Matrix:
[[ 2  1  0  1  3  5  0  8  0  1]
 [ 2  2  0  0  1  6  0 10  0  1]
 [ 1  1  0  0  1  6  0  6  0  0]
 [ 0  3  0  0  0  6  0  9  0  0]
 [ 3  2  0  0  2  5  0  5  0  0]
 [ 1  1  0  0  1  4  0  7  0  0]
 [ 1  1  0  0  1  6  1 11  0  0]
 [ 2  0  0  0  0 10  0  7  0  2]
 [ 0  2  0  0  2  6  0

In [33]:
# EXP-15: Perceptron - Breast Cancer Classification


from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Load Dataset
data = load_breast_cancer()

X = data.data
y = data.target

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Preprocessing
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create Perceptron Model
model = Perceptron(
    max_iter=1000,
    eta0=0.01,
    random_state=42
)

# Train Model
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)

# Display Output
print("Breast Cancer Classification Using Perceptron")
print("-----------------------------------------------")

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("\nClassification Accuracy: {:.2f}%".format(
    accuracy * 100
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[1, 0]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[1, 0],
    target_names=["Benign", "Malignant"]
))

# New Sample Prediction
sample = X_test[0].reshape(1, -1)

prediction = model.predict(sample)

print("New Sample Prediction:",
      "Benign" if prediction[0] == 1 else "Malignant")









Breast Cancer Classification Using Perceptron
-----------------------------------------------
Training Samples: 455
Testing Samples: 114

Classification Accuracy: 99.12%

Confusion Matrix:
[[71  0]
 [ 1 42]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.99      1.00      0.99        71
   Malignant       1.00      0.98      0.99        43

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114

New Sample Prediction: Benign


In [34]:
#EXPERIMENT-16:
import numpy as np

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Load Wine dataset
wine = load_wine()

X = wine.data
y = wine.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Build Naive Bayes model
model = GaussianNB()

# Train model
model.fit(X_train, y_train)

# Predict test data
y_pred = model.predict(X_test)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Wine Classification Using Naive Bayes")
print("--------------------------------------")

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("\nClassification Accuracy: {:.2f}%".format(
    accuracy * 100
))

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=wine.target_names
))

# Predict a new sample
new_sample = X_test[0].reshape(1, -1)

prediction = model.predict(new_sample)

print("\nNew Sample Prediction:",
      wine.target_names[prediction[0]])



Wine Classification Using Naive Bayes
--------------------------------------
Training Samples: 142
Testing Samples: 36

Classification Accuracy: 97.22%

Confusion Matrix:
[[12  0  0]
 [ 1 13  0]
 [ 0  0 10]]

Classification Report:
              precision    recall  f1-score   support

     class_0       0.92      1.00      0.96        12
     class_1       1.00      0.93      0.96        14
     class_2       1.00      1.00      1.00        10

    accuracy                           0.97        36
   macro avg       0.97      0.98      0.97        36
weighted avg       0.97      0.97      0.97        36


New Sample Prediction: class_0
